In [3]:
import os
import cv2
import torch
import numpy as np
import random
import warnings
warnings.filterwarnings('ignore')
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import albumentations as A
from tqdm import tqdm
model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 5)
print(model)

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

In [3]:
import os
import cv2
import torch
import numpy as np
import random
import warnings
warnings.filterwarnings('ignore')

from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import albumentations as A
from tqdm import tqdm

# =========================
# FIXED SEED
# =========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# =========================
# FREE GPU MEMORY
# =========================
torch.cuda.empty_cache()

# =========================
# CONFIG - OPTIMIZED FOR GTX 1660 6GB
# =========================
IMAGE_DIR = "train/images"
LABEL_DIR = "train/labels"
VAL_IMAGE_DIR = "valid/images"
VAL_LABEL_DIR = "valid/labels"

class_names = ["helmet", "head", "body", "vest"]
NUM_CLASSES = len(class_names) + 1

# Reduce batch size to prevent OOM
BATCH_SIZE = 2  # ← Reduced to 2
EPOCHS = 20

# Reduce image size to use less memory
IMAGE_SIZE = 640  # Or 512 for lower memory usage

BACKBONE_LAYER4_LR = 0.0005
RPN_HEAD_LR = 0.001
ROI_HEAD_LR = 0.001

WEIGHT_DECAY = 0.0005
MOMENTUM = 0.9

LR_FACTOR = 0.5
LR_PATIENCE = 3
MIN_LR = 1e-7

GRAD_CLIP_MAX_NORM = 1.0
EARLY_STOPPING_PATIENCE = 4

SAVE_DIR = "saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

LOAD_PRETRAINED = True
PRETRAINED_MODEL_PATH = "saved_models/faster_rcnn_final_old.pth"

# =========================
# AUGMENTATION with reduced image size
# =========================
transform = A.Compose(
    [
        A.Resize(height=IMAGE_SIZE, width=IMAGE_SIZE),  # ← Reduced image size
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.1),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, border_mode=cv2.BORDER_CONSTANT, p=0.5),
        A.Blur(blur_limit=3, p=0.1),
    ],
    bbox_params=A.BboxParams(
        format="pascal_voc",
        label_fields=["labels"],
        min_visibility=0.3,
        check_each_transform=True,
        min_area=25,
    )
)

val_transform = A.Compose(
    [
        A.Resize(height=IMAGE_SIZE, width=IMAGE_SIZE),  # ← Reduced image size
    ],
    bbox_params=A.BboxParams(format="pascal_voc", label_fields=["labels"])
)

# =========================
# DATASET
# =========================
class CustomDataset(Dataset):
    def __init__(self, image_dir, label_dir, image_list, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.image_list = image_list
        self.transform = transform

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_name = self.image_list[idx]
        img_path = os.path.join(self.image_dir, img_name)
        label_path = os.path.join(self.label_dir, img_name.replace(".jpg", ".txt"))

        image = cv2.imread(img_path)
        if image is None:
            image = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        height, width = image.shape[:2]

        boxes = []
        labels = []

        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls = int(parts[0])
                    xmin = float(parts[1])
                    ymin = float(parts[2])
                    xmax = float(parts[3])
                    ymax = float(parts[4])

                    xmin = max(0, min(width, xmin))
                    ymin = max(0, min(height, ymin))
                    xmax = max(0, min(width, xmax))
                    ymax = max(0, min(height, ymax))

                    if xmax <= xmin or ymax <= ymin:
                        continue

                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(cls + 1)

        if self.transform:
            if len(boxes) > 0:
                augmented = self.transform(image=image, bboxes=boxes, labels=labels)
                image = augmented["image"]
                boxes = augmented["bboxes"]
                labels = augmented["labels"]

                valid_boxes = []
                valid_labels = []
                for box, label in zip(boxes, labels):
                    if len(box) == 4:
                        xmin, ymin, xmax, ymax = box
                        if (xmax > xmin + 1 and ymax > ymin + 1 and 
                            xmin >= 0 and ymin >= 0 and 
                            xmax <= image.shape[1] and ymax <= image.shape[0]):
                            valid_boxes.append(box)
                            valid_labels.append(label)
                boxes = valid_boxes
                labels = valid_labels
            else:
                augmented = self.transform(image=image, bboxes=[], labels=[])
                image = augmented["image"]

        image = torch.from_numpy(image).float().permute(2, 0, 1) / 255.0

        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image = (image - mean) / std

        if len(boxes) == 0:
            target = {
                "boxes": torch.zeros((0, 4), dtype=torch.float32),
                "labels": torch.zeros((0,), dtype=torch.int64)
            }
        else:
            target = {
                "boxes": torch.tensor(boxes, dtype=torch.float32),
                "labels": torch.tensor(labels, dtype=torch.int64)
            }

        return image, target

# =========================
# DATA LOADING
# =========================
def filter_images_with_boxes(image_dir, label_dir, image_list):
    valid_images = []
    total_boxes = 0
    
    for img in image_list:
        label_path = os.path.join(label_dir, img.replace(".jpg", ".txt"))
        if os.path.exists(label_path):
            with open(label_path) as f:
                lines = f.readlines()
                if len(lines) > 0:
                    valid_images.append(img)
                    total_boxes += len(lines)
    
    print(f"  Found {len(valid_images)} valid images with {total_boxes} total boxes")
    return valid_images

train_images = [f for f in os.listdir(IMAGE_DIR) if f.endswith((".jpg", ".png", ".jpeg"))]
val_images = [f for f in os.listdir(VAL_IMAGE_DIR) if f.endswith((".jpg", ".png", ".jpeg"))]

print("Loading training data...")
train_images = filter_images_with_boxes(IMAGE_DIR, LABEL_DIR, train_images)

print("Loading validation data...")
val_images = filter_images_with_boxes(VAL_IMAGE_DIR, VAL_LABEL_DIR, val_images)

print(f"\nFinal counts:")
print(f"  Training samples: {len(train_images)}")
print(f"  Validation samples: {len(val_images)}")

def collate_fn(batch):
    return tuple(zip(*batch))

train_dataset = CustomDataset(IMAGE_DIR, LABEL_DIR, train_images, transform)
val_dataset = CustomDataset(VAL_IMAGE_DIR, VAL_LABEL_DIR, val_images, val_transform)

print("\nCreating DataLoaders...")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
    pin_memory=False
)

print("✅ DataLoaders created successfully!")

# =========================
# TEST DATALOADER
# =========================
print("\nTesting DataLoader...")
try:
    test_iter = iter(train_loader)
    test_batch = next(test_iter)
    print(f"✅ DataLoader works! Batch size: {len(test_batch[0])}")
    print(f"   Image shape: {test_batch[0][0].shape}")
except Exception as e:
    print(f"❌ DataLoader failed: {e}")
    exit()

# =========================
# MODEL
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

# Clear cache before model loading
torch.cuda.empty_cache()

model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

# =========================
# LOAD PRETRAINED WEIGHTS
# =========================
if LOAD_PRETRAINED and os.path.exists(PRETRAINED_MODEL_PATH):
    print("\n" + "="*70)
    print("LOADING PRETRAINED MODEL WEIGHTS")
    print("="*70)
    try:
        state_dict = torch.load(PRETRAINED_MODEL_PATH, map_location=device)
        model.load_state_dict(state_dict, strict=False)
        print(f"  ✅ Successfully loaded pretrained weights")
    except Exception as e:
        print(f"  ❌ Failed to load pretrained weights: {e}")
        LOAD_PRETRAINED = False

model.to(device)

# =========================
# LAYER FREEZING
# =========================
print("\n" + "="*70)
print("LAYER FREEZING CONFIGURATION")
print("="*70)

for param in model.parameters():
    param.requires_grad = False

print("\n📌 Backbone - Only layer4 fine-tuned:")
layer4_count = 0
for name, param in model.backbone.body.named_parameters():
    if 'layer4' in name:
        param.requires_grad = True
        layer4_count += 1
print(f"  Unfrozen: {layer4_count} parameters")

print("\n📌 RPN - ALL layers fine-tuned:")
rpn_count = 0
for param in model.rpn.parameters():
    param.requires_grad = True
    rpn_count += 1
print(f"  Unfrozen: {rpn_count} parameters")

print("\n📌 ROI Heads - ALL layers fine-tuned:")
roi_count = 0
for param in model.roi_heads.parameters():
    param.requires_grad = True
    roi_count += 1
print(f"  Unfrozen: {roi_count} parameters")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n" + "-"*70)
print(f"📊 Total parameters: {total_params:,}")
print(f"📊 Trainable parameters: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")
print("="*70 + "\n")

# =========================
# OPTIMIZER
# =========================
optimizer_params = [
    {
        'params': [p for n, p in model.backbone.body.named_parameters() 
                  if 'layer4' in n and p.requires_grad],
        'lr': BACKBONE_LAYER4_LR,
    },
    {
        'params': model.rpn.parameters(),
        'lr': RPN_HEAD_LR,
    },
    {
        'params': model.roi_heads.parameters(),
        'lr': ROI_HEAD_LR,
    },
]

optimizer = torch.optim.SGD(optimizer_params, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=LR_FACTOR, patience=LR_PATIENCE,
    threshold=1e-3, cooldown=2, min_lr=MIN_LR, verbose=True
)

metric = MeanAveragePrecision()

# =========================
# EARLY STOPPING
# =========================
class EarlyStopping:
    def __init__(self, patience=4, mode='max', min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = -float('inf') if mode == 'max' else float('inf')
        self.mode = mode
        self.early_stop = False

    def step(self, score):
        if self.mode == 'max':
            is_better = score > self.best_score + self.min_delta
        else:
            is_better = score < self.best_score - self.min_delta

        if is_better:
            self.best_score = score
            self.counter = 0
            return False
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                return True
        return False

    def get_status(self):
        return {
            'counter': self.counter,
            'patience': self.patience,
            'best_score': self.best_score,
            'remaining': self.patience - self.counter
        }

early_stopper = EarlyStopping(patience=EARLY_STOPPING_PATIENCE, mode='max', min_delta=0.001)

# =========================
# TRAINING LOOP
# =========================
print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70)
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Batch Size: {BATCH_SIZE} (Reduced for 6GB GPU)")
print(f"Image Size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Early Stopping Patience: {EARLY_STOPPING_PATIENCE} epochs")
print("="*70 + "\n")

best_map = 0
accumulation_steps = 2  # Gradient Accumulation to simulate batch_size=4

for epoch in range(EPOCHS):
    print(f"\n{'='*70}")
    print(f"🚀 EPOCH {epoch+1}/{EPOCHS}")
    print(f"{'='*70}")
    
    # -----------------
    # TRAIN PHASE
    # -----------------
    model.train()
    train_loss = 0
    train_losses = {}
    train_num_batches = 0
    
    # Clear cache at start of each epoch
    torch.cuda.empty_cache()

    pbar = tqdm(train_loader, desc=f"Training", unit='batch', 
                ncols=100, position=0, leave=True)
    
    optimizer.zero_grad()
    
    for batch_idx, (images, targets) in enumerate(pbar):
        try:
            # Move data to device
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # Forward pass
            loss_dict = model(images, targets)
            loss = sum(loss for loss in loss_dict.values())
            
            # Gradient Accumulation
            loss = loss / accumulation_steps
            loss.backward()

            # Track losses
            train_loss += loss.item() * accumulation_steps
            for k, v in loss_dict.items():
                if k not in train_losses:
                    train_losses[k] = 0
                train_losses[k] += v.item()
            train_num_batches += 1

            # Update weights every accumulation_steps
            if (batch_idx + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_MAX_NORM)
                optimizer.step()
                optimizer.zero_grad()

            # Update progress bar
            pbar.set_postfix({
                'loss': f'{loss.item() * accumulation_steps:.4f}',
                'lr': f'{optimizer.param_groups[0]["lr"]:.6f}'
            })

            # Check for NaN
            if torch.isnan(loss):
                print(f"\n❌ NaN detected at batch {batch_idx+1}!")
                raise RuntimeError("NaN detected!")
                
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"\n⚠️ CUDA OOM at batch {batch_idx}. Clearing cache...")
                torch.cuda.empty_cache()
                continue
            else:
                print(f"\n❌ Error in batch {batch_idx}: {e}")
                continue

    # Final optimizer step if needed
    if (batch_idx + 1) % accumulation_steps != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_MAX_NORM)
        optimizer.step()
        optimizer.zero_grad()

    pbar.close()

    # Calculate average losses
    train_loss_avg = train_loss / train_num_batches if train_num_batches > 0 else 0
    train_losses_avg = {k: v / train_num_batches for k, v in train_losses.items()}
    if train_num_batches > 0 else {}

    # -----------------
    # VALIDATION PHASE
    # -----------------
    print(f"\n🔄 Validating...")
    model.eval()
    metric.reset()

    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc="Validation", unit='batch', ncols=100, position=0, leave=True)
        for images, targets in val_pbar:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            preds = model(images)
            metric.update(preds, targets)
            
            val_pbar.set_postfix({'status': 'processing'})
        
        val_pbar.close()

    # Compute mAP
    results = metric.compute()
    current_map = results['map'].item()
    map_50 = results['map_50'].item()
    map_75 = results.get('map_75', torch.tensor(0.0)).item()

    # Update scheduler
    scheduler.step(current_map)

    # Early stopping check
    early_stop = early_stopper.step(current_map)
    status = early_stopper.get_status()

    # -----------------
    # PRINT RESULTS
    # -----------------
    print(f"\n📊 RESULTS EPOCH {epoch+1}:")
    print(f"  Training Loss: {train_loss_avg:.4f}")
    if train_losses_avg:
        for k, v in train_losses_avg.items():
            print(f"    - {k}: {v:.4f}")
    print(f"\n  Validation mAP:")
    print(f"    - mAP@0.5:0.95: {current_map:.4f}")
    print(f"    - mAP@0.5: {map_50:.4f}")
    print(f"    - mAP@0.75: {map_75:.4f}")
    print(f"\n  📊 Early Stopping:")
    print(f"    - Best mAP: {status['best_score']:.4f}")
    print(f"    - No improvement: {status['counter']}/{status['patience']}")
    print(f"    - Remaining: {status['remaining']}")

    # Save best model
    if current_map > best_map:
        best_map = current_map
        best_model_path = os.path.join(SAVE_DIR, "faster_rcnn_best.pth")
        torch.save(model.state_dict(), best_model_path)
        print(f"  ✅ New best model saved! (mAP: {best_map:.4f})")

    # Early stopping
    if early_stop:
        print("\n" + "="*70)
        print("🛑 EARLY STOPPING TRIGGERED!")
        print(f"  No improvement for {EARLY_STOPPING_PATIENCE} epochs")
        print(f"  Best mAP: {status['best_score']:.4f}")
        print("="*70)
        break

# =========================
# SAVE FINAL MODEL
# =========================
print("\n" + "="*70)
print("SAVING FINAL MODEL")
print("="*70)

final_model_path = os.path.join(SAVE_DIR, "faster_rcnn_final.pth")
torch.save(model.state_dict(), final_model_path)

file_size = os.path.getsize(final_model_path) / (1024 * 1024)
print(f"  💾 Final model saved: {final_model_path}")
print(f"  📁 File size: {file_size:.2f} MB")

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)

print(f"\n📈 SUMMARY:")
print(f"  Total epochs: {epoch+1}")
print(f"  Best mAP: {best_map:.4f}")
print(f"  Final mAP: {current_map:.4f}")
print(f"  Batch Size: {BATCH_SIZE} (with accumulation={accumulation_steps}, effective={BATCH_SIZE*accumulation_steps})")
print("="*70)

Loading training data...
  Found 1829 valid images with 4862 total boxes
Loading validation data...
  Found 457 valid images with 1176 total boxes

Final counts:
  Training samples: 1829
  Validation samples: 457

Creating DataLoaders...
✅ DataLoaders created successfully!

Testing DataLoader...
✅ DataLoader works! Batch size: 2
   Image shape: torch.Size([3, 640, 640])

Using device: cuda

LOADING PRETRAINED MODEL WEIGHTS
  ✅ Successfully loaded pretrained weights

LAYER FREEZING CONFIGURATION

📌 Backbone - Only layer4 fine-tuned:
  Unfrozen: 10 parameters

📌 RPN - ALL layers fine-tuned:
  Unfrozen: 6 parameters

📌 ROI Heads - ALL layers fine-tuned:
  Unfrozen: 8 parameters

----------------------------------------------------------------------
📊 Total parameters: 41,314,536
📊 Trainable parameters: 29,457,448 (71.30%)


STARTING TRAINING
GPU: NVIDIA GeForce GTX 1660 Ti
Batch Size: 2 (Reduced for 6GB GPU)
Image Size: 640x640
Early Stopping Patience: 4 epochs


🚀 EPOCH 1/20


Training: 100%|██████████████████████| 915/915 [05:09<00:00,  2.96batch/s, loss=0.2990, lr=0.000500]



🔄 Validating...


Validation: 100%|███████████████████████████| 229/229 [00:54<00:00,  4.19batch/s, status=processing]



📊 RESULTS EPOCH 1:
  Training Loss: 0.4723
    - loss_classifier: 0.2250
    - loss_box_reg: 0.1573
    - loss_objectness: 0.0741
    - loss_rpn_box_reg: 0.0159

  Validation mAP:
    - mAP@0.5:0.95: 0.0552
    - mAP@0.5: 0.1344
    - mAP@0.75: 0.0339

  📊 Early Stopping:
    - Best mAP: 0.0552
    - No improvement: 0/4
    - Remaining: 4
  ✅ New best model saved! (mAP: 0.0552)

🚀 EPOCH 2/20


...


📊 RESULTS EPOCH 19:
  Training Loss: 0.3419
    - loss_classifier: 0.1557
    - loss_box_reg: 0.1469
    - loss_objectness: 0.0299
    - loss_rpn_box_reg: 0.0094

  Validation mAP:
    - mAP@0.5:0.95: 0.2323
    - mAP@0.5: 0.4274
    - mAP@0.75: 0.2352

  📊 Early Stopping:
    - Best mAP: 0.2343
    - No improvement: 2/4
    - Remaining: 2

🚀 EPOCH 20/20


Training: 100%|██████████████████████| 915/915 [05:06<00:00,  2.98batch/s, loss=0.8090, lr=0.000500]



🔄 Validating...


Validation: 100%|███████████████████████████| 229/229 [00:52<00:00,  4.33batch/s, status=processing]



📊 RESULTS EPOCH 20:
  Training Loss: 0.3407
    - loss_classifier: 0.1562
    - loss_box_reg: 0.1483
    - loss_objectness: 0.0274
    - loss_rpn_box_reg: 0.0089

  Validation mAP:
    - mAP@0.5:0.95: 0.2464
    - mAP@0.5: 0.4476
    - mAP@0.75: 0.2485

  📊 Early Stopping:
    - Best mAP: 0.2464
    - No improvement: 0/4
    - Remaining: 4
  ✅ New best model saved! (mAP: 0.2464)

SAVING FINAL MODEL
  💾 Final model saved: saved_models\faster_rcnn_final.pth
  📁 File size: 158.11 MB

TRAINING COMPLETE!

📈 SUMMARY:
  Total epochs: 20
  Best mAP: 0.2464
  Final mAP: 0.2464
  Batch Size: 2 (with accumulation=2, effective=4)
